In [1]:
import torch
import torch.nn as nn
from torchvision import models
import time
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

In [2]:
# we first check if the GPU is active

In [3]:
print(torch.cuda.is_available())

True


In [4]:
print(torch.__version__)

2.12.0.dev20260408+cu128


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [6]:
# since these variables and functions from notebook 2 are in use by this notebook,
# we rewrite them here

In [7]:
LABELS = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
          'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
          'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other',
          'Pneumonia', 'Pneumothorax', 'Support Devices']

IMG_DIR = r"D:\omer files\projects\NMIMS\cxr_project"

In [8]:
class CXRDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform = None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        def build_path(row):
            subj = f"p{str(row['subject_id'])[:2]}/p{row['subject_id']}"
            study = f"s{row['study_id']}"
            return os.path.join('files', subj, study, f"{row['dicom_id']}.jpg")
        self.df['rel_path'] = self.df.apply(build_path, axis=1)

        self.df['full_path'] = self.df['rel_path'].apply(lambda p: os.path.join(img_dir, p))
        before = len(self.df)
        self.df = self.df[self.df['full_path'].apply(os.path.exists)].reset_index(drop = True)
        print(f"{csv_path}: {len(self.df)}/{before} images found on disk")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row[LABELS].values.astype('float32'))
        return img, labels

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness = 0.1, contrast = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [11]:
train_ds = CXRDataset('subset_train.csv', IMG_DIR, transform = train_transform)
val_ds = CXRDataset('subset_val.csv', IMG_DIR, transform = eval_transform)
test_ds = CXRDataset('subset_test.csv', IMG_DIR, transform = eval_transform)

train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_ds, batch_size = 32, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = False, num_workers = 0)

subset_train.csv: 5537/5537 images found on disk
subset_val.csv: 300/300 images found on disk
subset_test.csv: 300/300 images found on disk


In [12]:
imgs, labels = next(iter(train_loader))
print(f'Batch image shape: {imgs.shape}')
print(f'Batch label shape: {labels.shape}')
print(f'Label sum per sample (first 5): {labels[:5].sum(dim = 1)}')

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32, 14])
Label sum per sample (first 5): tensor([5., 5., 5., 1., 2.])


In [13]:
# we now build the model

In [14]:
# we now load the ResNet50 model and replace the final layer for the 14-label multi-label output
model = models.resnet50(weights = models.ResNet50_Weights.IMAGENET1K_V2)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(LABELS))    # 14 outputs, no sigmoid here
model = model.to(device)

In [15]:
for name, param in model.named_parameters():
    if not name.startswith('layer4') and not name.startswith('fc'):
        param.requires_grad = False

In [18]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable,} / {total:,}')

Trainable params: (14993422,) / 23,536,718


In [19]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr = 1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 2)

In [20]:
# these are the train/validate functions

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

In [22]:
# we initialize the training loops with early stopping

In [23]:
NUM_EPOCHS = 15
EARLY_STOP_PATIENCE = 4

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    elapsed = time.time() - start

    print(f"Epoch {epoch+1} / {NUM_EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f" -> Saved new best model (val_loss={val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping — no improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

print('Training complete. Best val loss: ', best_val_loss)

Epoch 1 / 15 | Train Loss: 0.4633 | Val Loss: 0.3390 | Time: 337.1s
 -> Saved new best model (val_loss=0.3390)
Epoch 2 / 15 | Train Loss: 0.4153 | Val Loss: 0.3330 | Time: 320.3s
 -> Saved new best model (val_loss=0.3330)
Epoch 3 / 15 | Train Loss: 0.3966 | Val Loss: 0.3175 | Time: 305.5s
 -> Saved new best model (val_loss=0.3175)
Epoch 4 / 15 | Train Loss: 0.3785 | Val Loss: 0.3233 | Time: 422.8s
Epoch 5 / 15 | Train Loss: 0.3587 | Val Loss: 0.3202 | Time: 404.8s
Epoch 6 / 15 | Train Loss: 0.3359 | Val Loss: 0.3369 | Time: 410.4s
Epoch 7 / 15 | Train Loss: 0.3022 | Val Loss: 0.3574 | Time: 366.2s
Early stopping — no improvement for 4 epochs.
Training complete. Best val loss:  0.31747632344563803
